# 모델 비교
- 4주 평균 VS 랜덤포레스트(합쳐서) VS 랜덤포레스트(상품별)

In [1]:
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 작업 경로를 프로젝트 최상위 폴더로 맞춘다.
# → 프로젝트 내의 파이썬 모듈 문제없이 import
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from datetime import datetime, timezone

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sqlalchemy import select

from app import models
from app.database import SessionLocal
from app.repositories import ForecastRepository
from app.forecasting import FEATURE_COLS, active_week_ratio, build_table, wape, weekly_sales

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

In [2]:
# 1) DB에서 주문 행 가져오기
with SessionLocal() as db:
    lines = pd.DataFrame(ForecastRepository().order_lines(db), columns=['product_id', 'ordered_at', 'quantity'])
    sku = {p.id: p.sku for p in db.scalars(select(models.Product))}
assert not lines.empty, "DB에 주문이 없습니다. dummy_orders.py로 만든 CSV를 먼저 /imports/orders로 올려주세요."

# 2) '주 X 상품' 수량 표로 바꾸기
now = datetime.now(timezone.utc).replace(tzinfo=None)
weekly = weekly_sales(lines, now)
print(f'주간 표: {weekly.shape[0]}주 x {weekly.shape[1]}개 상품 ({weekly.index.min().date()} ~ {weekly.index.max().date()})')
weekly.head()

주간 표: 52주 x 12개 상품 (2025-10-06 ~ 2026-09-28)


,1,2,3,4,5,6,7,8,9,10,11,12
week_start,,,,,,,,,,,,
2025-10-06,232.0,958.0,576.0,15.0,1277.0,0.0,481.0,149.0,36.0,302.0,64.0,4.0
2025-10-13,138.0,364.0,273.0,2.0,159.0,5.0,303.0,174.0,39.0,104.0,38.0,5.0
2025-10-20,432.0,563.0,359.0,8.0,706.0,6.0,204.0,216.0,45.0,229.0,29.0,15.0
2025-10-27,489.0,668.0,306.0,5.0,569.0,2.0,166.0,92.0,66.0,282.0,19.0,1.0
2025-11-03,188.0,663.0,334.0,3.0,766.0,11.0,257.0,164.0,21.0,229.0,1.0,0.0


## 1. 예측 대상 상품만 고르기
- "최근 52주 중 주문 있던 주 비율"이 0.8(임시 기준) 미만이면 예측 대상에서 제외한다.

In [3]:
MIN_RATIO = 0.8

ratio = active_week_ratio(weekly)
eligible = list(ratio[ratio >= MIN_RATIO].index)
we = weekly[eligible]
print('예측 대상:', [sku[p] for p in eligible])
print('제외     :', [sku[p] for p in weekly.columns if p not in eligible])

예측 대상: ['P-004', 'P-003', 'P-011', 'P-010', 'P-005', 'P-001', 'P-012', 'P-008', 'P-002', 'P-009']
제외     : ['P-007', 'P-006']


In [4]:
table = build_table(we)
table.assign(sku=table['product_id'].map(sku)).sample(5)

,product_id,week_start,target,lag_1,lag_2,lag_3,lag_4,avg_4,month_start_days,sku
469,11,2026-07-20,12.0,34.0,21.0,21.0,11.0,21.75,0,P-009
123,3,2026-05-11,410.0,397.0,239.0,361.0,454.0,362.75,0,P-011
386,10,2025-11-17,221.0,247.0,229.0,282.0,229.0,246.75,0,P-002
233,6,2026-08-17,0.0,6.0,1.0,6.0,4.0,4.25,0,P-005
375,9,2026-08-03,79.0,46.0,24.0,24.0,46.0,35.00,5,P-008


## 2. 비교 함수
- 마지막 8주 → 테스트

In [ ]:
RF_PARAMS = dict(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)

def make_pooled_model():
    """
    입력값 6개는 그대로 통과시키고, 'product_id'만 원-핫으로 바꿔서 랜덤포레스트에 넣는다.
    """
    pre = ColumnTransformer([
        ('features', 'passthrough', FEATURE_COLS),
        ('product', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['product_id'])
    ], verbose_feature_names_out=False)
    return Pipeline([('pre', pre), ('rf', RandomForestRegressor(**RF_PARAMS))])

def backtest(weekly, holdout_weeks=8, per_product=False):
    """
    마지막 holdout_weeks주를 테스트로 떼어두고, 그 주들의 '다음 1주'를 한 주씩 맞혀본다.

    반환값
    - 한 줄 = (상품, 주)
    - target이 정답이고, 방법별 예측값이 컬럼
    """
    table = build_table(weekly)
    cut = weekly.index[-holdout_weeks]
    train, test = table[table['week_start'] < cut], table[table['week_start'] >= cut]
    out = test[['product_id', 'week_start', 'target']].copy()

    # 비교 1: 최근 4주 평균을 그대로 다음 주 예측으로 사용
    out['moving_average_4'] = test['avg_4']

    # 비교 2: 랜덤포레스트 → 모든 상품을 한 표로 합쳐서 학습
    pooled = make_pooled_model().fit(train, train['target'])
    out['random_forest_pooled'] = pooled.predict(test)

    # 비교 3: 랜덤포레스트 → 상품별로 따로 학습
    if per_product:
        out['random_forest_per_product'] = 0.0
        for pid, g in train.groupby('product_id'):
            model = RandomForestRegressor(**RF_PARAMS).fit(g[FEATURE_COLS], g['target'])
            is_this_product = test['product_id'] == pid
            out.loc[is_this_product, 'random_forest_per_product'] = model.predict(test.loc[is_this_product, FEATURE_COLS])
    return out

def wape_by_product(res, model):
    """
    백테스트 결과 테이블에서 model 열의 예측 오차(WAPE)를 상품별로 계산해
    {상품 id: WAPE} 형태의 Series로 돌려준다.
    
    - res: 실제값과 모델들이 예측한 값들이 들어가 있는 결과 테이블
    - model: 평가하고 싶은 특정 모델의 예측 결과 컬럼 이름
    """
    return pd.Series({pid: wape(g['target'], g[model]) for pid, g in res.groupby('product_id')})

In [6]:
# 원-핫 결과
pre = make_pooled_model().named_steps['pre'].fit(table)
sample = table.sample(5, random_state=0)
pd.DataFrame(pre.transform(sample), columns=pre.get_feature_names_out(), index=sample.index).round(2)

,lag_1,lag_2,lag_3,lag_4,avg_4,month_start_days,product_id_1,product_id_2,product_id_3,product_id_5,product_id_6,product_id_7,product_id_8,product_id_9,product_id_10,product_id_11
15,209.0,295.0,162.0,252.0,229.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
124,410.0,397.0,239.0,361.0,351.75,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
141,323.0,369.0,596.0,68.0,339.00,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
263,234.0,232.0,264.0,336.0,266.50,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
170,365.0,928.0,829.0,1008.0,782.50,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. 다음 1주 예측 비교
- 전체: 판매량이 큰 상품의 오차가 더 크게 반영된다.
- 상품 평균: 모든 상품을 똑같은 비중으로 본다.

In [9]:
models_ = ['moving_average_4', 'random_forest_pooled', 'random_forest_per_product']

res = backtest(we, holdout_weeks=8, per_product=True)

per_prod = pd.DataFrame({m: wape_by_product(res, m) for m in models_})
per_prod.index = [sku[p] for p in per_prod.index]

overall = pd.DataFrame({m: {'전체(판매량 가중)': wape(res['target'], res[m])} for m in models_}).round(3)
mean_row = per_prod.mean().to_frame('상품 평균').T

pd.concat([per_prod, overall, mean_row]).round(3)

,moving_average_4,random_forest_pooled,random_forest_per_product
P-004,0.460,0.345,0.389
P-003,0.439,0.455,0.385
P-011,0.420,0.403,0.381
P-010,0.380,0.354,0.329
P-005,0.716,0.550,0.556
P-001,0.297,0.219,0.223
P-012,0.674,0.624,0.717
P-008,0.575,0.653,0.691
P-002,0.257,0.282,0.379
P-009,0.595,0.674,0.676


## 4. 테스트 기간에 따른 결과 비교

In [10]:
rows = {}
for h in (8, 12, 16):
    r = backtest(we, holdout_weeks=h, per_product=True)
    rows[f"테스트 마지막 {h}주"] = {m: wape(r["target"], r[m]) for m in models_}
pd.DataFrame(rows).T.round(3)

,moving_average_4,random_forest_pooled,random_forest_per_product
테스트 마지막 8주,0.405,0.381,0.378
테스트 마지막 12주,0.398,0.399,0.378
테스트 마지막 16주,0.415,0.398,0.383


## 5. 모델 선택 결과
| 구간 | 4주 평균 | 합쳐서 | 상품별 |
| --- | --- | --- | --- |
| 8주 | 0.405 | 0.381 | 0.378 |
| 12주 | 0.398 | 0.399 | 0.378 |
| 16주 | 0.415 | 0.398 | 0.383 |

- 상품별 랜덤포레스트가 세 구간 모두에서 가장 낮은 오차를 보였고, 4주 평균 대비 약 5~8% 개선됐다.
- 합쳐서 학습한 모델은 구간에 따라 4주 평균과 비슷하거나 소폭 개선에 그쳐 일관성이 부족했다.
- 따라서 상품별 랜덤포레스트를 채택했다.